# Quadratic Hawkes (exp synapse): pipeline theory vs simulation

Unified comparison notebook for the standardised quad-Hawkes theory
(`pipeline.theories.quad_hawkes_2pop_expg`).  Same plumbing for any
`k` / `max_ell` combination; the plot + residual cells dispatch on
`k`:

- **k = 1** (rate / single-leg cumulant): bar chart of sim vs `n*`
  (tree) vs `n* + tadpole` (tree + 1-loop) per population.
- **k = 2** (cross-cumulant): overlay of the simulated factorial
  $C^{(2)}(\tau)$ with the theory tree (and optionally tree + loop)
  curve.

Theory side is a single call to `pipeline.compute_cumulants(...)`;
simulation side uses `models.hawkes_sim_quad_expg_numba` directly and
the factorial cumulant estimator from `models.cumulant_estimator`.
Swap the `theory` import to a different `pipeline.theories.*` module
and you get the corresponding sim/theory comparison for free.

## 1. Setup

In [ ]:
%display latex
%matplotlib inline
%load_ext autoreload
%autoreload 2

import os, sys, time
import numpy as np
import matplotlib.pyplot as plt

# Pipeline (theory side)
sys.path.insert(0, os.path.abspath('..'))
from pipeline import compute_cumulants
from pipeline.theories.quad_hawkes_2pop_expg import HAWKES_MODEL

# Sim side
from models.hawkes_sim_quad_expg_numba import sim_hawkes_quad_expg_numba
from models.cumulant_estimator import compute_kpoint_slice

## 2. Configuration

Theory + simulation share the `fundamental` dict.  Pick `k` (1 or 2)
and `max_ell` (0 for tree-only, 1 to engage the cubic-vertex 1-loop
diagrams).  For `k = 1` we recommend `max_ell = 1` — the tree-level
rate is just `n*` (the MF saddle); the loop correction is the
tadpole shift.  For `k = 2`, both `max_ell = 0` (LNA tree level)
and `max_ell = 1` (tree + 1-loop) are useful comparisons.

In [ ]:
fundamental = {
    'E':     [0.5, 0.5],
    'w':     [[0.3, 0.25], [0.3, 0.35]],
    'tau':   10.0,
    'a':     0.44,
    'tau_g': 5.0,
}

# k-point cumulant + loop order
k        = 2
max_ell  = 0

# External fields — auto-resolved by k.
# For k=1, pass a single tuple (which population's rate to extract).
# For k=2, pass two tuples (which two legs).
if k == 1:
    external_fields = [('n', 1)]
else:
    external_fields = [('n', 1), ('n', 2)]

# τ grid (theory + sim share this).
# k=1: stationary rates are τ-independent, so a coarse grid is fine
#      (we use it to assert constancy).
# k=2: pick tau_max large enough to capture the cumulant tail.
if k == 1:
    tau_max  = 20.0
    tau_step = 5.0
else:
    tau_max  = 50.0
    tau_step = 0.5

# Pipeline parallelism — fanned across enumeration (step [5]) and
# Phase J τ-grid evaluation (step [7]).  Set N_WORKERS=None to let
# the pipeline pick min(os.cpu_count(), n_tasks).  PARALLEL=False
# forces serial (useful for debugging).
PARALLEL  = True
N_WORKERS = None

# ─── Grouped Phase J prototype (opt-in) ───────────────────────
# When True, groups typed diagrams by (parent prediagram,
# external_legs) and integrates the SUMMED integrand once per group
# instead of once per typed diagram.  Mathematically exact (bitwise
# cross-checked).  Speedup scales with the typed-to-prediagram
# ratio — modest at k=2 ell=0 (few typed diagrams), larger at higher
# loop orders where the per-prediagram diagram count balloons.
USE_GROUPED_PHASE_J = False

# Simulation
N_RUNS   = 5
T_sim    = float(5_000_000)    # total time per run (rate × T sets the
                                # signal-to-shot-noise ratio of the
                                # estimator)
dt_sim   = 0.01                 # Euler step
dt_bin   = 0.25                 # binning resolution for the k=2 cumulant
                                # estimator (ignored for k=1; rates are
                                # bin-resolution independent)

print(f'k={k}, max_ell={max_ell}, external_fields={external_fields}')
print(f'tau_max={tau_max}, tau_step={tau_step}')
print(f'PARALLEL={PARALLEL}, N_WORKERS={N_WORKERS}, '
      f'USE_GROUPED_PHASE_J={USE_GROUPED_PHASE_J}')
print(f'N_RUNS={N_RUNS}, T_sim={T_sim:.0g}, dt_sim={dt_sim}, dt_bin={dt_bin}')

## 3. Theory side — one pipeline call

`compute_cumulants` runs the full MSR-JD pipeline: FieldTheory
expansion → propagator construction → MF self-consistency → diagram
enumeration / typing / dedup → Phase J integration on the τ grid.

For `k = 1`, the τ-grid output is τ-independent (it's a 1-point
function), so we cross-check that the spread across τ is ≪ 1 as a
sanity check, then take the mean as the rate-correction value.

In [ ]:
t0 = time.perf_counter()
th = compute_cumulants(
    model               = HAWKES_MODEL,
    k                   = k,
    max_ell             = max_ell,
    fundamental         = fundamental,
    external_fields     = external_fields,
    tau_max             = tau_max,
    tau_step            = tau_step,
    parallel            = PARALLEL,
    n_workers           = N_WORKERS,
    use_grouped_phase_j = USE_GROUPED_PHASE_J,
    verbose             = True,
)
print(f'\nTheory side took {time.perf_counter() - t0:.1f}s')

tau_grid_th    = th['tau_grid']
C_theory_total = th['C_tau'].real
C_by_ell       = th['C_tau_by_ell']     # {0: tree, 1: 1-loop, ...}
C_theory_tree  = (C_by_ell[0].real if 0 in C_by_ell
                  else np.zeros_like(C_theory_total))
C_theory_loop  = C_theory_total - C_theory_tree

mf    = th['mf']
nstar = np.array(mf['n'], dtype=float)
vstar = np.array(mf['v'], dtype=float)
npop_th = len(nstar)

print(f'\nMF: ' + ', '.join(f'{k_}={mf[k_]}' for k_ in mf))
for i in range(npop_th):
    print(f'   pop {i+1}:  n* = {mf["n", i+1]:.6f},  '
          f'v* = {mf["v", i+1]:.6f}')
print(f'  Total diagrams: {len(th["diagrams"])}')
n_per_ell = {ell: sum(1 for r in th['diagrams'] if r['ell'] == ell)
             for ell in sorted({r['ell'] for r in th['diagrams']})}
for ell, n_d in n_per_ell.items():
    print(f'    ell={ell}: {n_d} diagrams')

if USE_GROUPED_PHASE_J:
    from collections import Counter
    grp_counts = Counter()
    for ell, td_res in th['phase_j_by_ell'].items():
        if td_res is None:
            continue
        for _ in td_res.get('groups', []):
            grp_counts[ell] += 1
    print(f'  Grouped Phase J: prediagram groups per ell = '
          f'{dict(grp_counts)}')

# ── k=1 post-processing: per-population rate corrections ──────
if k == 1:
    # Stationarity sanity check: C_total should be ~constant across τ.
    spread = (float(np.max(np.abs(C_theory_total - C_theory_total.mean()))) /
              max(abs(C_theory_total.mean()), 1e-12))
    print(f'\n  τ-grid relative spread (should be ≪ 1): {spread:.2e}')

    ext_pop_idx = external_fields[0][1] - 1   # 1-based → 0-based
    # Total loop correction = sum across all ell ≥ 1 (= total - tree).
    # Tree is identically zero at the saddle for k=1, but we compute
    # it explicitly for robustness.
    total_loop = float(C_theory_total.mean())
    loop_correction = np.zeros_like(nstar)
    loop_correction[ext_pop_idx] = total_loop

    rate_theory_tree     = nstar.copy()
    rate_theory_treeloop = nstar + loop_correction

    print(f'\n  Per-ell mean values (population {ext_pop_idx+1}):')
    for ell in sorted(C_by_ell.keys()):
        val = float(C_by_ell[ell].mean().real)
        label = 'tree' if ell == 0 else f'{ell}-loop'
        print(f'    C_{label}: {val:+.6e}')
    print(f'\n  rate_tree (n*):       {rate_theory_tree}')
    print(f'  rate_tree+loop:       {rate_theory_treeloop}')

## 4. Simulation side

Same Numba simulator either way.  For `k = 1` we just collect
per-population firing rates `total_spikes / T_sim`; for `k = 2` we
also run `compute_kpoint_slice` to get the factorial covariance
$C^{(2)}(\tau)$.

In [ ]:
import secrets as _secrets

E_sim     = [float(x) for x in fundamental['E']]
w_sim     = [[float(x) for x in row] for row in fundamental['w']]
tau_sim   = float(fundamental['tau'])
tau_g_sim = float(fundamental['tau_g'])
a_sim     = float(fundamental['a'])
npop_sim  = len(E_sim)

n_steps        = int(T_sim / dt_sim)
bin_size_steps = max(int(round(dt_bin / dt_sim)), 1)
dt_bin_eff     = bin_size_steps * dt_sim
n_bins         = n_steps // bin_size_steps
max_lag_bins   = int(tau_max / dt_bin_eff)
tau_sim_grid   = np.arange(-max_lag_bins, max_lag_bins + 1) * dt_bin_eff

v_init = np.array(vstar, dtype=float)
E_arr  = np.array(E_sim,  dtype=float)
W_arr  = np.array(w_sim,  dtype=float)

# JIT warmup
_ = sim_hawkes_quad_expg_numba(
    int(1000), float(dt_sim), float(tau_sim),
    float(tau_g_sim), float(a_sim),
    E_arr, W_arr, v_init.copy(),
    int(bin_size_steps), int(100), int(0),
)

BASE_SEED   = _secrets.randbits(31)
pop_indices = [ef[1] - 1 for ef in external_fields]
field_types = [ef[0] for ef in external_fields]

C_sim_runs = []      # only populated for k>=2
rate_runs  = []
t0 = time.perf_counter()
for run in range(N_RUNS):
    seed = int(BASE_SEED + run)
    binned_counts, voltage_bins, total_spikes = sim_hawkes_quad_expg_numba(
        int(n_steps), float(dt_sim), float(tau_sim),
        float(tau_g_sim), float(a_sim),
        E_arr, W_arr, v_init.copy(),
        int(bin_size_steps), int(n_bins), seed,
    )
    rate_runs.append(
        [float(total_spikes[i]) / T_sim for i in range(npop_sim)]
    )
    if k == 1:
        # k=1 cumulant ≡ time-averaged rate; total_spikes/T_sim already
        # estimates it.  Skip the 2-point slice machinery.
        pass
    else:
        tau_run, C_run = compute_kpoint_slice(
            binned_counts, float(dt_bin_eff),
            [int(p) for p in pop_indices],
            [0, None], int(max_lag_bins),
            field_types=field_types,
            voltage_bins=voltage_bins,
        )
        C_sim_runs.append(C_run)
    print(f'  run {run+1}/{N_RUNS}: rates = '
          f'{[f"{r:.4f}" for r in rate_runs[-1]]}')

if k >= 2:
    C_sim_runs = np.array(C_sim_runs)
    C_sim_mean = C_sim_runs.mean(axis=0)
    C_sim_sem  = (C_sim_runs.std(axis=0, ddof=1) / np.sqrt(N_RUNS)
                  if N_RUNS > 1 else np.zeros_like(C_sim_mean))
else:
    C_sim_runs = C_sim_mean = C_sim_sem = None

rate_runs_arr   = np.array(rate_runs)
rate_sim_mean   = rate_runs_arr.mean(axis=0)
rate_sim_sem    = (rate_runs_arr.std(axis=0, ddof=1) / np.sqrt(N_RUNS)
                   if N_RUNS > 1 else np.zeros_like(rate_sim_mean))

print(f'\nSimulation side took {time.perf_counter() - t0:.1f}s '
      f'({N_RUNS} runs × T={T_sim:.0g})')
print(f'  Sim mean rates ± SEM:')
for i in range(npop_sim):
    print(f'    pop {i+1}:  {rate_sim_mean[i]:.6f} ± {rate_sim_sem[i]:.6f}')
print(f'  Theory n*:        {nstar}')

## 5. Theory vs simulation

**For k = 1**: per-population bar chart with three bars per population
(simulation, theory tree = `n*`, theory tree + 1-loop = `n* + tadpole`).
Only the configured external population gets the loop correction —
the others show their tree bar in place of the loop bar.  A converged
1-loop pipeline brings the orange bar (loop-corrected) onto the green
bar (sim) within the SEM.

**For k = 2**: side-by-side mean firing rates (sim vs MF) on the
left, plus the $C^{(2)}(\tau)$ slice on the right with the tree and
(if `max_ell > 0`) tree + loop overlays.  The sim shading is the
across-run SEM band.

In [ ]:
if k == 1:
    # ── k=1: rate bar chart per population ──
    fig, ax = plt.subplots(figsize=(8, 5))
    x = np.arange(npop_sim)
    width = 0.27

    ax.bar(x - width, rate_theory_tree, width,
           label='Theory: tree (n*)',
           color='#3498DB', edgecolor='black')
    ax.bar(x, rate_theory_treeloop, width,
           label=f'Theory: tree + {max_ell}-loop',
           color='#F39C12', edgecolor='black')
    ax.bar(x + width, rate_sim_mean, width, yerr=rate_sim_sem,
           label='Simulation', color='#2ECC71', edgecolor='black',
           capsize=4)

    # Annotate which pop got the loop correction.
    if max_ell >= 1:
        ax.annotate(
            f'1-loop computed for pop {external_fields[0][1]}',
            xy=(ext_pop_idx, rate_theory_treeloop[ext_pop_idx]),
            xytext=(ext_pop_idx + 0.4,
                    rate_theory_treeloop[ext_pop_idx] * 1.08),
            fontsize=9,
            arrowprops=dict(arrowstyle='->', color='#555'),
        )

    ax.set_xticks(x)
    ax.set_xticklabels([f'Pop {i+1}' for i in range(npop_sim)])
    ax.set_ylabel('Firing rate')
    ax.set_title(f'k=1 rate: tree vs 1-loop vs simulation '
                 f'(quad Hawkes, T={T_sim:.0g}, N_RUNS={N_RUNS})')
    ax.legend()
    ax.grid(True, axis='y', alpha=0.25)
    fig.tight_layout()
    plt.show()
else:
    # ── k=2: rate bars + C^(2)(τ) overlay ──
    fig, axes = plt.subplots(1, 2, figsize=(14, 5),
                             gridspec_kw={'width_ratios': [1, 2]})

    ax_bar = axes[0]
    x = np.arange(npop_sim)
    width = 0.35
    ax_bar.bar(x - width/2, rate_sim_mean, width,
               yerr=rate_sim_sem, capsize=3,
               label='Simulation', color='#2ECC71', edgecolor='black')
    ax_bar.bar(x + width/2, nstar, width,
               label='Mean-field', color='#3498DB', edgecolor='black')
    ax_bar.set_xticks(x)
    ax_bar.set_xticklabels([f'Pop {i+1}' for i in range(npop_sim)])
    ax_bar.set_ylabel('Firing rate')
    ax_bar.set_title('Mean firing rates')
    ax_bar.legend()
    ax_bar.grid(True, axis='y', alpha=0.25)

    ax = axes[1]
    ax.plot(tau_sim_grid, C_sim_mean, color='#1f1f1f', linewidth=1.4,
            label=f'Simulation ({N_RUNS} runs avg)', alpha=0.85)
    ax.fill_between(tau_sim_grid,
                    C_sim_mean - C_sim_sem,
                    C_sim_mean + C_sim_sem,
                    color='#1f1f1f', alpha=0.15, label='Sim SEM')
    ax.plot(tau_grid_th, C_theory_tree, color='#3F00FF', linewidth=1.6,
            linestyle='--', label='Theory: tree only')
    if max_ell > 0:
        ax.plot(tau_grid_th, C_theory_total, color='#E74C3C',
                linewidth=1.6,
                label=f'Theory: tree + {max_ell}-loop')
    ax.axhline(0, color='gray', linewidth=0.5)
    ax.axvline(0, color='gray', linewidth=0.5)
    field_a, field_b = external_fields
    ax.set_xlabel(r'$\tau_1$')
    ax.set_ylabel(r'$C^{(2)}$')
    ax.set_title(f'Cross-cumulant: '
                 f'$\\langle\\delta {field_a[0]}_{{{field_a[1]}}}(0)\\,'
                 f'\\delta {field_b[0]}_{{{field_b[1]}}}(\\tau)\\rangle_c$')
    ax.legend()
    ax.grid(True, alpha=0.25)
    fig.tight_layout()
    plt.show()

## 6. Numerical residual

**k = 1**: per-population residual table (sim − tree, sim − tree+loop)
with the residual reported in units of the sim SEM.  A converged
1-loop residual should be within ±2 σ.

**k = 2**: RMS / max-abs residual of `sim − (tree + loop)` over the
τ-grid, relative to the sim peak.  If `residual ≈ sim_SEM at peak`,
theory and sim agree within Monte-Carlo noise.

In [ ]:
if k == 1:
    print(f'{"pop":>4} {"sim ± SEM":>22} {"tree (n*)":>14} '
          f'{"tree+loop":>14} {"loop Δ":>14} '
          f'{"resid (sim-loop)/SEM":>22}')
    print('-' * 96)
    for i in range(npop_sim):
        sem = rate_sim_sem[i]
        resid = rate_sim_mean[i] - rate_theory_treeloop[i]
        resid_over_sem = (abs(resid) / sem) if sem > 0 else float('nan')
        print(f'{i+1:>4d} '
              f'{rate_sim_mean[i]:>10.6f} ± {sem:.6f}  '
              f'{rate_theory_tree[i]:>14.6f} '
              f'{rate_theory_treeloop[i]:>14.6f} '
              f'{loop_correction[i]:>+14.6e} '
              f'{resid_over_sem:>22.2f}')
    print('\n(If max_ell >= 1, the loop-corrected residual should be '
          'within ±2σ for the population we computed the loop for.)')
else:
    C_total_on_sim_grid = np.interp(tau_sim_grid, tau_grid_th,
                                    C_theory_total)
    residual            = C_sim_mean - C_total_on_sim_grid

    peak        = max(abs(C_sim_mean.max()), abs(C_sim_mean.min()))
    rms_rel     = float(np.sqrt(np.mean(residual**2)) / peak)
    max_abs_rel = float(np.max(np.abs(residual)) / peak)
    sem_peak    = float(C_sim_sem[np.argmax(np.abs(C_sim_mean))])

    print(f'Sim peak |C|             : {peak:+.4e}')
    print(f'Residual RMS / peak      : {rms_rel:.3%}')
    print(f'Residual max / peak      : {max_abs_rel:.3%}')
    print(f'Sim SEM at peak          : {sem_peak:+.3e}')
    print('(if residual ≈ sim SEM at peak, theory and sim agree '
          'within sim noise)')

    if max_ell > 0:
        C_tree_on_sim_grid = np.interp(tau_sim_grid, tau_grid_th,
                                       C_theory_tree)
        tree_residual      = C_sim_mean - C_tree_on_sim_grid
        tree_rms_rel       = float(np.sqrt(np.mean(tree_residual**2))
                                   / peak)
        print()
        print(f'Tree-only residual RMS / peak  : {tree_rms_rel:.3%}')
        print(f'Tree+loop residual RMS / peak  : {rms_rel:.3%}'
              f'   ← Δ = {tree_rms_rel - rms_rel:+.3%}')
        print('(positive Δ ⇒ loop correction shrank the residual)')

## 7. (Optional) Save NPZ + PDF report

In [ ]:
from pipeline import save_npz, save_csv

SAVE = False    # flip to True when you're happy with the run

if SAVE:
    out_dir = '../pipeline_outputs/quad_expg_sim_compare'
    os.makedirs(out_dir, exist_ok=True)
    leg_tag = '_'.join(f'{ef[0]}{ef[1]}' for ef in external_fields)
    slug = f'quad_expg_{leg_tag}_k{k}_ell{max_ell}'

    sim_extra = {
        'rates_sim_mean' : rate_sim_mean,
        'rates_sim_sem'  : rate_sim_sem,
        'sim_N_RUNS'     : np.array([N_RUNS], dtype=int),
        'sim_T'          : np.array([T_sim]),
        'sim_dt'         : np.array([dt_sim]),
        'sim_dt_bin'     : np.array([dt_bin]),
    }
    if k >= 2:
        sim_extra.update({
            'tau_grid_sim' : tau_sim_grid,
            'C_sim_mean'   : C_sim_mean,
            'C_sim_sem'    : C_sim_sem,
        })
    npz_path = f'{out_dir}/{slug}.npz'
    csv_path = f'{out_dir}/{slug}.csv'
    save_npz(th, npz_path, extra=sim_extra)
    save_csv(th, csv_path)
    print(f'Saved: {npz_path}')
    print(f'Saved: {csv_path}')

    from pipeline import generate_report
    pdf_path = f'{out_dir}/{slug}_report.pdf'
    generate_report(
        model           = HAWKES_MODEL,
        k               = k,
        fundamental     = fundamental,
        external_fields = external_fields,
        output_pdf      = pdf_path,
        result          = th,
        verbose         = False,
    )
    print(f'Saved: {pdf_path}')
else:
    print('SAVE=False — outputs not written.  Flip the flag above to save.')

---

### Switching k / ell / parameters

All numerical knobs live in section 2.  Three modes are supported:

- **k = 1, max_ell = 1** — tadpole rate correction.  Bar chart
  comparison; `external_fields` is a single tuple naming the
  population whose rate gets the loop correction.
- **k = 2, max_ell = 0** — tree-level (LNA) $C^{(2)}(\tau)$ slice.
- **k = 2, max_ell = 1** — tree + 1-loop $C^{(2)}(\tau)$ slice;
  the simulation should track the tree + loop curve, NOT the
  tree-only curve.

### Grouped Phase J prototype

Set `USE_GROUPED_PHASE_J = True` in section 2 to engage the grouped
Phase J path (groups typed diagrams by parent prediagram, integrates
the summed integrand once per group).  Mathematically exact;
bitwise cross-checked.  Speedup is structural — proportional to the
typed-to-prediagram ratio.  For this quad-expg theory at
`k=2, max_ell=0` the ratio is small; bump `max_ell = 1` to see the
grouping pay off.